Sudoku CSP Solver
This program solves Sudoku using CSP techniques like backtracking,
forward checking, and AC-3 for constraint propagation.

Reads a 9x9 Sudoku board from a text file.Each line contains digits and 0 represents an empty cell.



In [ ]:
import copy
from collections import deque


def read_board(filename):
    board = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line:
                row = [int(c) for c in line]
                board.append(row)
    return board

For a given cell, this function returns all other cells
that are in the same row, column, or 3x3 box.
These are the cells that must have different values.

In [ ]:
def get_peers(row, col):
    peers = set()

    # Add cells from the same row
    for c in range(9):
        if c != col:
            peers.add((row, c))

    # Add cells from the same column
    for r in range(9):
        if r != row:
            peers.add((r, col))

    # Add cells from the same 3x3 box
    box_r = (row // 3) * 3
    box_c = (col // 3) * 3

    for r in range(box_r, box_r + 3):
        for c in range(box_c, box_c + 3):
            if (r, c) != (row, col):
                peers.add((r, c))

    return peers


# Precompute peers for all cells so we don't calculate them again and again
ALL_PEERS = {}
for r in range(9):
    for c in range(9):
        ALL_PEERS[(r, c)] = get_peers(r, c)


Creates the domain for each cell.
If the cell already has a value, its domain is just that value.
Otherwise, the domain starts as numbers from 1 to 9.
Applies the AC-3 algorithm to make domains consistent.
It removes values that cannot satisfy constraints with neighboring cells.

In [ ]:
def build_domains(board):
  
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] != 0:
                domains[(r, c)] = {board[r][c]}
            else:
                domains[(r, c)] = set(range(1, 10))
    return domains


def ac3(domains):
    queue = deque()

    # Add all arcs (cell, its peer) to the queue
    for cell in domains:
        for peer in ALL_PEERS[cell]:
            queue.append((cell, peer))

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            # If domain becomes empty, no solution is possible
            if len(domains[xi]) == 0:
                return False

            # If we changed xi, we need to recheck its neighbors
            for xk in ALL_PEERS[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True

In [ ]:
This function checks if values in xi need to be removed
based on the current domain of xj.
In Sudoku, if xj is fixed to one value, xi cannot take that value.

In [ ]:
def revise(domains, xi, xj):
    revised = False

    if len(domains[xj]) == 1:
        val = next(iter(domains[xj]))

        if val in domains[xi]:
            domains[xi].discard(val)
            revised = True

    return revised


Uses MRV (Minimum Remaining Values) heuristic.
It selects the cell with the smallest domain first
to reduce the search space.
After assigning a value to a cell, this removes that value
from all its neighboring cells (peers).
If any neighbor loses all possible values, we stop early.

In [ ]:
def select_unassigned_variable(domains, assigned):
    
    unassigned = [cell for cell in domains if cell not in assigned]
    return min(unassigned, key=lambda cell: len(domains[cell]))


def forward_check(domains, cell, value):
    for peer in ALL_PEERS[cell]:
        if value in domains[peer]:
            domains[peer].discard(value)

            if len(domains[peer]) == 0:
                return False

    return True

This is the main backtracking function.
It tries assigning values to cells and recursively solves the puzzle.
If a choice leads to a dead end, it backtracks.

In [ ]:
# Counters to track performance
backtrack_calls = 0
backtrack_failures = 0


def backtrack(domains, assigned):
    global backtrack_calls, backtrack_failures
    backtrack_calls += 1

    # If all cells are assigned, solution is complete
    if len(assigned) == 81:
        return assigned

    # Choose next variable using MRV
    cell = select_unassigned_variable(domains, assigned)

    # Try values one by one
    for value in sorted(domains[cell]):

        # Copy domains so changes don't affect other branches
        domains_copy = copy.deepcopy(domains)

        assigned[cell] = value
        domains_copy[cell] = {value}

        # Apply forward checking
        if forward_check(domains_copy, cell, value):

            # Further reduce domains using AC-3
            if ac3(domains_copy):

                result = backtrack(domains_copy, assigned)

                if result is not None:
                    return result

        # Undo assignment if it didn't work
        del assigned[cell]

    backtrack_failures += 1
    return None


Sets up the CSP and starts solving.
Also keeps track of how many times backtracking was used.


In [ ]:
def solve(board):
    global backtrack_calls, backtrack_failures
    backtrack_calls = 0
    backtrack_failures = 0

    domains = build_domains(board)

    # First apply AC-3 to simplify problem
    if not ac3(domains):
        return None, backtrack_calls, backtrack_failures

    # Assign already filled cells
    assigned = {}
    for cell in domains:
        r, c = cell
        if board[r][c] != 0:
            assigned[cell] = board[r][c]

    solution = backtrack(domains, assigned)

    if solution is None:
        return None, backtrack_calls, backtrack_failures

    # Convert solution into board form
    result = [[0] * 9 for _ in range(9)]
    for (r, c), val in solution.items():
        result[r][c] = val

    return result, backtrack_calls, backtrack_failures

Prints the board in a simple grid format.

In [ ]:
def print_board(board):
    for r in range(9):
        for c in range(9):
            print(board[r][c], end=" ")
        print()

In [ ]:
def main():
    puzzles = [
        ("easy.txt", "Easy"),
        ("medium.txt", "Medium"),
        ("hard.txt", "Hard"),
        ("veryhard.txt", "Very Hard"),
    ]

    for filename, label in puzzles:
        print(label)
        print("Input:")

        board = read_board(filename)
        print_board(board)

        solved, calls, failures = solve(board)

        if solved:
            print("Solution:")
            print_board(solved)
        else:
            print("No solution")

        print("Backtrack calls:", calls)
        print("Backtrack failures:", failures)

        if label == "Easy":
            print("Most values are fixed early so very little backtracking is needed")
        elif label == "Medium":
            print("Some guessing is required but still manageable")
        elif label == "Hard":
            print("Search becomes deeper and more backtracking happens")
        elif label == "Very Hard":
            print("Large search space so many failures and retries occur")


if __name__ == "__main__":
    main()